# My List

The original **26 human genes in seven functional categories** are retained exactly. Functional categories are not exclusive M1/M2 identities. All available genes are included in descriptive summaries, regardless of expression direction or detection percentage; missing genes remain in the tables with NA statistics.

The shared setup reads `three_sample_macrophage_CAR_renamed.rds`, maps CAR/CAR_M/CARM to CARM and CAR_DR18/DR18/DR18_M to DR18, validates both groups, joins RNA layers, and applies LogNormalize with scale factor 10,000 and seed 42. New CSVs, the input MD5 checksum, parameters, and R session information are saved only in `results/carm_dr18_two_panels`; historical scripts and results are unchanged.

This is a **pooled-cell descriptive comparison**, not biological-replicate inference. Exploratory two-sided, unpaired Student t-tests assume equal variances and treat cells as observations. Tests use linear normalized expression including zeros. Raw p-values and Benjamini–Hochberg adjusted p-values (within each gene panel) are exported. These p-values do not establish biological-replicate significance; zero-variance comparisons are marked undefined. Positive `log2((DR18_mean_normalized + 1) / (CARM_mean_normalized + 1))` indicates higher mean expression in DR18; negative indicates CARM. Means and detection percentages are reported for each group. A gene missing from RNA is distinct from an available gene with zero detected cells.

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 6, width = 220, max.print = 100000)
required <- c('Seurat', 'SeuratObject', 'ggplot2', 'patchwork', 'Matrix', 'IRdisplay')
suppressPackageStartupMessages({
  for (package in required) {
    if (!requireNamespace(package, quietly = TRUE)) stop('Missing package: ', package)
    library(package, character.only = TRUE)
  }
})
stopifnot(utils::packageVersion('Seurat') >= '5.0.0', utils::packageVersion('SeuratObject') >= '5.0.0')
seed <- 42L
set.seed(seed)
input_file <- 'three_sample_macrophage_CAR_renamed.rds'
output_dir <- 'results/carm_dr18_two_panels'
stopifnot(file.exists(input_file))
dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)
object <- readRDS(input_file)
stopifnot(inherits(object, 'Seurat'), 'RNA' %in% names(object@assays), 'orig.ident' %in% names(object@meta.data))
group_map <- c(CAR = 'CARM', CAR_M = 'CARM', CARM = 'CARM', CAR_DR18 = 'DR18', DR18 = 'DR18', DR18_M = 'DR18')
labels <- as.character(object@meta.data$orig.ident)
groups <- unname(group_map[labels])
if (anyNA(groups)) stop('Unmapped groups: ', paste(unique(labels[is.na(groups)]), collapse = ', '))
object$comparison_group <- factor(groups, levels = c('CARM', 'DR18'))
stopifnot(all(table(object$comparison_group) > 0))
SeuratObject::DefaultAssay(object) <- 'RNA'
object <- SeuratObject::JoinLayers(object, assay = 'RNA')
object <- Seurat::NormalizeData(object, assay = 'RNA', normalization.method = 'LogNormalize', scale.factor = 10000, verbose = FALSE)
rna <- SeuratObject::LayerData(object, assay = 'RNA', layer = 'data')
stopifnot(identical(colnames(rna), colnames(object)))
colors <- c(CARM = '#2878A5', DR18 = '#CE5151')
cell_counts <- data.frame(comparison_group = levels(object$comparison_group), cells = as.integer(table(object$comparison_group)))
utils::write.csv(cell_counts, file.path(output_dir, 'cell_counts.csv'), row.names = FALSE)
print(cell_counts, row.names = FALSE)
parameters <- list(input = normalizePath(input_file), input_md5 = tools::md5sum(input_file), seed = seed,
                   groups = group_map, species_symbols = 'human', normalization = 'RNA LogNormalize',
                   scale_factor = 10000, directional_filtering = FALSE, colors = colors,
                   font_size = 14,
                   interpretation = 'Pooled-cell descriptive comparison; no replicate-level inference or p-values')
saveRDS(parameters, file.path(output_dir, 'analysis_parameters.rds'))
capture.output(utils::sessionInfo(), file = file.path(output_dir, 'sessionInfo.txt'))

display_table <- function(values) {
  displayed <- values
  numeric_columns <- vapply(displayed, is.numeric, logical(1))
  displayed[numeric_columns] <- lapply(displayed[numeric_columns], round, digits = 4)
  print(displayed, row.names = FALSE)
}
markdown_table <- function(values) {
  text <- rbind(names(values), rep('---', ncol(values)), as.matrix(values))
  text[is.na(text)] <- 'NA'
  IRdisplay::display_markdown(paste(apply(text, 1, function(row) paste0('| ', paste(row, collapse = ' | '), ' |')), collapse = '\n'))
}
summarize_panel <- function(catalog, prefix) {
  catalog$available <- catalog$gene %in% rownames(rna)
  summary <- catalog
  stat_columns <- c('CARM_mean_log1p', 'CARM_mean_normalized', 'CARM_detected_pct',
                    'DR18_mean_log1p', 'DR18_mean_normalized', 'DR18_detected_pct', 'log2fc_DR18_vs_CARM')
  stat_columns <- c(stat_columns, unlist(lapply(c('CARM', 'DR18'), function(g) paste0(g, c('_sd_normalized', '_sem_normalized', '_n_cells')))))
  for (column_name in stat_columns) summary[[column_name]] <- NA_real_
  summary$higher_group <- NA_character_
  summary$status <- ifelse(summary$available, 'present_not_detected', 'missing_from_RNA')
  if (any(summary$available)) {
    expression <- rna[summary$gene[summary$available], , drop = FALSE]
    for (group_name in c('CARM', 'DR18')) {
      values <- expression[, object$comparison_group == group_name, drop = FALSE]
      summary[summary$available, paste0(group_name, '_mean_log1p')] <- Matrix::rowMeans(values)
      summary[summary$available, paste0(group_name, '_mean_normalized')] <- Matrix::rowMeans(expm1(values))
      normalized_values <- as.matrix(expm1(values))
      cell_sd <- apply(normalized_values, 1, stats::sd)
      summary[summary$available, paste0(group_name, '_sd_normalized')] <- cell_sd
      summary[summary$available, paste0(group_name, '_sem_normalized')] <- cell_sd / sqrt(ncol(values))
      summary[summary$available, paste0(group_name, '_n_cells')] <- ncol(values)
      summary[summary$available, paste0(group_name, '_detected_pct')] <- 100 * Matrix::rowMeans(values > 0)
    }
    summary$log2fc_DR18_vs_CARM <- with(summary, log2((DR18_mean_normalized + 1) / (CARM_mean_normalized + 1)))
    summary$higher_group <- with(summary, ifelse(log2fc_DR18_vs_CARM > 0, 'DR18', ifelse(log2fc_DR18_vs_CARM < 0, 'CARM', 'Equal')))
    detected <- summary$available & (summary$CARM_detected_pct > 0 | summary$DR18_detected_pct > 0)
    summary$status[which(detected)] <- 'present_detected'
  }
  # Exploratory cell-level Student's t-test: equal variances, unpaired, two-sided.
  # Test linear normalized expression, matching the bars; all cells including zeros.
  summary$t_statistic <- summary$df <- summary$p_value <- NA_real_
  summary$test_status <- ifelse(summary$available, 'not_tested', 'missing_from_RNA')
  for (i in which(summary$available)) {
    x <- as.numeric(expm1(rna[summary$gene[i], object$comparison_group == 'CARM']))
    y <- as.numeric(expm1(rna[summary$gene[i], object$comparison_group == 'DR18']))
    result <- tryCatch(stats::t.test(x, y, alternative = 'two.sided', paired = FALSE,
                                     var.equal = TRUE), error = function(e) e)
    if (inherits(result, 'error')) {
      summary$test_status[i] <- paste0('undefined: ', conditionMessage(result))
    } else {
      summary$t_statistic[i] <- unname(result$statistic)
      summary$df[i] <- unname(result$parameter)
      summary$p_value[i] <- result$p.value
      summary$test_status[i] <- 'tested'
    }
  }
  summary$p_adj_BH <- stats::p.adjust(summary$p_value, method = 'BH')
  summary$test_method <- 'Exploratory cell-level two-sided unpaired Student t-test (equal variances)'
  summary$test_scale <- 'Linear normalized RNA expression; includes zero-expression cells'
  summary$adjustment_family <- paste0('BH within ', prefix, ' panel')
  utils::write.csv(summary, file.path(output_dir, paste0(prefix, '_student_t_tests.csv')), row.names = FALSE)
  utils::write.csv(summary, file.path(output_dir, paste0(prefix, '_gene_summary.csv')), row.names = FALSE)
  utils::write.csv(summary[!summary$available, , drop = FALSE], file.path(output_dir, paste0(prefix, '_missing_genes.csv')), row.names = FALSE)
  cat(sprintf('%s: %d resolved genes; %d available; %d missing; %d present but undetected in both groups.\n',
              prefix, nrow(summary), sum(summary$available), sum(!summary$available), sum(summary$status == 'present_not_detected')))
  display_table(summary)
  markdown_table(summary[, intersect(c('requested', 'gene', 'category', 'mapping', 'available', 'status'), names(summary)), drop = FALSE])
  summary
}

In [ ]:
my_list <- list(
  Mono_derived = c('FCGR3A', 'NFKB1', 'CSF1R'),
  Chemotaxis = c('CSF1', 'CCL18', 'CXCL9'),
  Phagocytic_Mac = c('C1QA', 'C1QB', 'C1QC'),
  MHC_I = c('HLA-A', 'HLA-B', 'HLA-C'),
  MHC_II = c('HLA-DRA', 'HLA-DRB1', 'HLA-DQA1', 'HLA-DQB1', 'HLA-DMA', 'HLA-DMB', 'CD74'),
  Ag_processing = c('TAP1', 'TAP2', 'TAPBP', 'PSMB9', 'B2M'),
  Co_stim = c('CD86', 'CD40')
)
my_genes <- unlist(my_list, use.names = FALSE)
stopifnot(length(my_genes) == 26L, !anyDuplicated(my_genes))
my_catalog <- data.frame(gene = my_genes, category = rep(names(my_list), lengths(my_list)))
my_summary <- summarize_panel(my_catalog, 'my_list')
parameters$my_list <- my_list
saveRDS(parameters, file.path(output_dir, 'analysis_parameters.rds'))
# Gene-level bar charts: pooled mean normalized RNA expression.
source("my_list_barplots.R")
my_combined_barplot <- make_combined_my_list_barplot(my_summary, output_dir, colors)
options(repr.plot.width = 14, repr.plot.height = 20.5)
print(my_combined_barplot)


# Exploration List

Requested order: **VEGFA, NFKB1, IL1B, IL6, CD80, CD86, TLR4, IL12, CXCL9, CXCL10, CXCL11, iNOS, MRC1, CD163, MACRO, CD274, LRIG2, IFITM1, IFITM2, IRF1, IFIT2, ARG, TGFB1, VSIG4, IL13**.

All available resolved human symbols are included in the descriptive summary, regardless of expression direction or detection percentage. VEGFA and NFKB1 are explicitly included; sharing NFKB1, CD86, or CXCL9 with My List is intentional. No exclusive M1/M2 labels are assigned.

Explicit human-symbol resolution:

| Requested name | Resolved human symbol(s) | Interpretation |
| --- | --- | --- |
| MACRO | MARCO | Likely typo, explicitly interpreted as MARCO rather than silently replaced |
| iNOS | NOS2 | Common protein/enzyme name mapped to human gene symbol |
| IL12 | IL12A, IL12B | Both IL-12 subunit genes; not a single IL12 gene |
| ARG | ARG1, ARG2 | Both arginase genes; ARG is ambiguous |
| All other names | Unchanged | Literal human gene symbols |

The code below expands the 25 requested entries to 27 resolved symbols in the same order. The complete mapping and observed availability/status table is rendered as Markdown and exported as `exploration_alias_mapping.csv`. Both expression-summary CSVs retain full precision. `missing_from_RNA` means absent from the RNA feature set; `present_not_detected` means available but detected in neither group. Neither state is silently omitted from the tables.

In [ ]:
exploration_requested <- c('VEGFA', 'NFKB1', 'IL1B', 'IL6', 'CD80', 'CD86', 'TLR4', 'IL12',
  'CXCL9', 'CXCL10', 'CXCL11', 'iNOS', 'MRC1', 'CD163', 'MACRO', 'CD274', 'LRIG2',
  'IFITM1', 'IFITM2', 'IRF1', 'IFIT2', 'ARG', 'TGFB1', 'VSIG4', 'IL13')
aliases <- list(IL12 = c('IL12A', 'IL12B'), iNOS = 'NOS2', MACRO = 'MARCO', ARG = c('ARG1', 'ARG2'))
mapping_notes <- c(IL12 = 'Expanded to both IL-12 subunit genes', iNOS = 'Common name mapped to NOS2',
                   MACRO = 'Likely typo interpreted as MARCO', ARG = 'Ambiguous name expanded to both arginases')
exploration_catalog <- do.call(rbind, lapply(seq_along(exploration_requested), function(request_index) {
  requested <- exploration_requested[[request_index]]
  resolved <- if (requested %in% names(aliases)) aliases[[requested]] else requested
  note <- if (requested %in% names(mapping_notes)) unname(mapping_notes[[requested]]) else 'Literal human symbol'
  data.frame(request_order = request_index, requested = requested, gene = resolved, mapping = note)
}))
rownames(exploration_catalog) <- NULL
stopifnot(length(exploration_requested) == 25L, nrow(exploration_catalog) == 27L,
          !anyDuplicated(exploration_catalog$gene), identical(head(exploration_catalog$gene, 2), c('VEGFA', 'NFKB1')))
exploration_summary <- summarize_panel(exploration_catalog, 'exploration')
exploration_mapping <- exploration_summary[, c('request_order', 'requested', 'gene', 'mapping', 'available', 'status')]
utils::write.csv(exploration_mapping, file.path(output_dir, 'exploration_alias_mapping.csv'), row.names = FALSE)
parameters$exploration_requested <- exploration_requested
parameters$exploration_resolved <- exploration_catalog
parameters$alias_mapping <- aliases
saveRDS(parameters, file.path(output_dir, 'analysis_parameters.rds'))
cat('\nVEGFA and NFKB1 descriptive expression:\n')
display_table(exploration_summary[exploration_summary$gene %in% c('VEGFA', 'NFKB1'), , drop = FALSE])
# Gene-level bar charts: pooled mean normalized RNA expression.
source("my_list_barplots.R")
exploration_barplots <- make_my_list_barplots(exploration_summary, output_dir, colors,
                                            prefix = 'exploration', title = 'Exploration List')
for (p in exploration_barplots) print(p)


In [ ]:
cat("\nReproducibility parameters:\n")
print(parameters)
capture.output(utils::sessionInfo(), file = file.path(output_dir, 'sessionInfo.txt'))
print(utils::sessionInfo())
cat("\nCurrent summary and bar-chart exports:\n")
current_exports <- list.files(output_dir, full.names = TRUE)
image_file <- grepl("\\.(pdf|png|jpe?g|svg)$", basename(current_exports), ignore.case = TRUE)
bar_chart_file <- grepl("bar", basename(current_exports), ignore.case = TRUE)
print(current_exports[!image_file | bar_chart_file])

In [ ]:
# WP_IL18_SIGNALING per-cell signature score (Seurat AddModuleScore; not sample-level GSEA)
wp_records <- msigdbr::msigdbr(
  species = "Homo sapiens",
  collection = "C2",
  subcollection = "CP:WIKIPATHWAYS"
)
wp_il18_genes <- unique(wp_records$gene_symbol[wp_records$gs_name == "WP_IL18_SIGNALING"])
stopifnot(length(wp_il18_genes) > 0L)
wp_present <- intersect(wp_il18_genes, rownames(rna))
wp_missing <- setdiff(wp_il18_genes, rownames(rna))
stopifnot(length(wp_present) > 0L)

wp_gene_audit <- data.frame(
  gene = wp_il18_genes,
  available_in_RNA = wp_il18_genes %in% rownames(rna)
)
utils::write.csv(wp_gene_audit, file.path(output_dir, "WP_IL18_SIGNALING_gene_audit.csv"), row.names = FALSE)
cat(sprintf("WP_IL18_SIGNALING: %d signature genes; %d in RNA; %d absent.\n",
            length(wp_il18_genes), length(wp_present), length(wp_missing)))

scored_object <- Seurat::AddModuleScore(
  object = object,
  features = list(wp_present),
  assay = "RNA",
  name = "WP_IL18_SIGNALING_",
  slot = "data",
  seed = seed
)
wp_score_column <- "WP_IL18_SIGNALING_1"
stopifnot(wp_score_column %in% colnames(scored_object[[]]))
wp_score_data <- data.frame(
  cell = colnames(scored_object),
  group = scored_object$comparison_group,
  score = scored_object[[]][[wp_score_column]]
)
wp_score_summary <- do.call(rbind, lapply(levels(wp_score_data$group), function(group_name) {
  scores <- wp_score_data$score[wp_score_data$group == group_name]
  data.frame(group = group_name, n_cells = length(scores), mean_score = mean(scores),
             median_score = median(scores), sd_score = sd(scores))
}))
rownames(wp_score_summary) <- NULL
utils::write.csv(wp_score_summary, file.path(output_dir, "WP_IL18_SIGNALING_score_by_group.csv"), row.names = FALSE)
utils::write.csv(wp_score_data, file.path(output_dir, "WP_IL18_SIGNALING_score_per_cell.csv"), row.names = FALSE)
cat("\nSeurat AddModuleScore by group (pooled cells; exploratory Student t-test):\n")
display_table(wp_score_summary)
# Signature means +/- SEM and exploratory cell-level Student t-test.
source("signature_barplots.R")
wp_plot_data <- wp_score_data
names(wp_plot_data)[names(wp_plot_data) == "score"] <- "WP_IL18_SIGNALING"
wp_stats <- signature_statistics(wp_plot_data, "WP_IL18_SIGNALING", output_dir, "WP_IL18_SIGNALING")
wp_score_bar_plot <- plot_signature_statistics(wp_stats, output_dir,
  "WP_IL18_SIGNALING_mean_score", colors, "WikiPathways IL-18 signature score", ncol = 1)
options(repr.plot.width = 6.5, repr.plot.height = 5.5)
print(wp_score_bar_plot)


In [ ]:
# Additional MSigDB signatures that reflect IL-18 response, production, or related pathways.
# These are per-cell Seurat AddModuleScores, not sample-level GSEA enrichment tests.
reactome_records <- msigdbr::msigdbr(
  species = "Homo sapiens", collection = "C2", subcollection = "CP:REACTOME"
)
go_bp_records <- msigdbr::msigdbr(
  species = "Homo sapiens", collection = "C5", subcollection = "GO:BP"
)
il18_signature_specs <- data.frame(
  signature = c(
    "REACTOME_INTERLEUKIN_18_SIGNALING",
    "GOBP_RESPONSE_TO_INTERLEUKIN_18",
    "GOBP_INTERLEUKIN_18_PRODUCTION",
    "REACTOME_THE_NLRP3_INFLAMMASOME",
    "REACTOME_INFLAMMASOMES",
    "REACTOME_INTERFERON_GAMMA_SIGNALING"
  ),
  source = c("C2:CP:REACTOME", "C5:GO:BP", "C5:GO:BP", "C2:CP:REACTOME", "C2:CP:REACTOME", "C2:CP:REACTOME"),
  stringsAsFactors = FALSE
)
il18_signature_records <- rbind(reactome_records, go_bp_records)
il18_signature_genes <- setNames(lapply(il18_signature_specs$signature, function(signature_name) {
  unique(il18_signature_records$gene_symbol[il18_signature_records$gs_name == signature_name])
}), il18_signature_specs$signature)
stopifnot(all(lengths(il18_signature_genes) > 0L))
il18_signature_matched <- lapply(il18_signature_genes, intersect, y = rownames(rna))
stopifnot(all(lengths(il18_signature_matched) > 0L))

il18_signature_audit <- do.call(rbind, lapply(names(il18_signature_genes), function(signature_name) {
  genes <- il18_signature_genes[[signature_name]]
  data.frame(signature = signature_name, gene = genes, available_in_RNA = genes %in% rownames(rna))
}))
utils::write.csv(il18_signature_audit,
                 file.path(output_dir, "IL18_related_signature_gene_audit.csv"), row.names = FALSE)
cat("MSigDB signature coverage:\n")
print(data.frame(
  signature = names(il18_signature_genes),
  source = il18_signature_specs$source,
  genes_in_set = lengths(il18_signature_genes),
  genes_in_RNA = lengths(il18_signature_matched)
), row.names = FALSE)

il18_related_object <- Seurat::AddModuleScore(
  object = object,
  features = unname(il18_signature_matched),
  assay = "RNA",
  name = "IL18_related_",
  slot = "data",
  seed = seed
)
il18_score_columns <- setNames(
  paste0("IL18_related_", seq_along(il18_signature_genes)), names(il18_signature_genes)
)
stopifnot(all(il18_score_columns %in% colnames(il18_related_object[[]])))
il18_score_per_cell <- data.frame(
  cell = colnames(il18_related_object),
  group = il18_related_object$comparison_group
)
for (signature_name in names(il18_signature_genes)) {
  il18_score_per_cell[[signature_name]] <- il18_related_object[[]][[il18_score_columns[[signature_name]]]]
}
utils::write.csv(il18_score_per_cell,
                 file.path(output_dir, "IL18_related_signature_scores_per_cell.csv"), row.names = FALSE)
il18_score_summary <- do.call(rbind, lapply(names(il18_signature_genes), function(signature_name) {
  do.call(rbind, lapply(levels(il18_score_per_cell$group), function(group_name) {
    scores <- il18_score_per_cell[[signature_name]][il18_score_per_cell$group == group_name]
    data.frame(
      signature = signature_name,
      source = il18_signature_specs$source[match(signature_name, il18_signature_specs$signature)],
      genes_in_set = length(il18_signature_genes[[signature_name]]),
      genes_in_RNA = length(il18_signature_matched[[signature_name]]),
      group = group_name,
      n_cells = length(scores),
      mean_score = mean(scores),
      median_score = median(scores),
      sd_score = sd(scores)
    )
  }))
}))
rownames(il18_score_summary) <- NULL
utils::write.csv(il18_score_summary,
                 file.path(output_dir, "IL18_related_signature_scores_by_group.csv"), row.names = FALSE)
cat("\nAdditional signature scores by group (pooled cells; exploratory Student t-tests):\n")
display_table(il18_score_summary)

# Signature means +/- SEM; raw p-values of cell-level unpaired Student t-tests.
source("signature_barplots.R")
il18_stats <- signature_statistics(il18_score_per_cell, names(il18_signature_genes),
  output_dir, "IL18_related_signature")
il18_score_bar_plot <- plot_signature_statistics(il18_stats, output_dir,
  "IL18_related_signature_mean_scores", colors)
options(repr.plot.width = 9.6, repr.plot.height = 11.2)
print(il18_score_bar_plot)
